In [ ]:
"""
simulation.py

Main driver for the latest OS-DML simulation study, sequentially executing
three distinct experiments. All content is in English.
"""
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

import os
import pickle
import traceback
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import numpy as np

import config
import evaluation

# Suppress LightGBM output
os.environ['LIGHTGBM_VERBOSITY'] = '-1'

In [2]:
def run_single_replication(params):
    """Runs one full simulation replication for a given set of parameters."""
    exp_name, scenario_name, method_name, sim_id, specific_params = params
    
    try:
        seed = config.BASE_SEED + sim_id
        np.random.seed(seed)
        
        scenarios, all_methods, experiments = config.get_experiments()
        
        scenario_config = scenarios[scenario_name]
        method_config = all_methods[method_name]
        
        # Generate data
        data = scenario_config['data_gen_func'](**scenario_config['params'])
        true_ate = data['true_ate']
        is_rct = (scenario_config['design'] == 'rct')
        
        # Run the method
        func = method_config['func']
        
        # Combine general and experiment-specific parameters
        run_params = {**method_config.get('params', {}), **specific_params}
        
        result_dict = func(
            X=data['X'], W=data['W'], Y_obs=data['Y_obs'], 
            pi_true=data['pi_true'], is_rct=is_rct, **run_params
        )
        
        # Add metadata and save the result
        result_dict.update({
            'scenario': scenario_name, 'sim_id': sim_id, 'method': method_name,
            'true_ate': true_ate, **specific_params
        })
        
        results_dir = os.path.join(experiments[exp_name]['base_dir'], scenario_name)
        os.makedirs(results_dir, exist_ok=True)
        
        # Create a unique filename for the specific parameters
        param_str = "_".join(f"{k}_{v}".replace('.', 'p') for k, v in specific_params.items() if k not in ['r', 'k_folds'])
        if 'r' in specific_params:
            param_str += f"_r0_{specific_params['r']['r0']}_r1_{specific_params['r']['r1']}"
        
        param_str = param_str.replace(" ", "_").replace("'", "")
        result_file = os.path.join(results_dir, f"sim_{sim_id}_method_{method_name}_{param_str}.pkl")

        with open(result_file, 'wb') as f:
            pickle.dump(result_dict, f)

    except Exception:
        print(f"\n--- ERROR in worker: Exp={exp_name}, Scenario={scenario_name}, Method={method_name}, Sim ID={sim_id} ---")
        traceback.print_exc()
        print("-" * 70)

In [3]:
scenarios, all_methods, experiments = config.get_experiments()

for exp_name, exp_config in experiments.items():
    print(f"\n🚀 === Running: {exp_name} ===")
    print(f"   {exp_config['description']}")
    
    tasks = []
    
    # Construct task list for the current experiment
    if exp_name == "experiment_1_pilot_allocation":
        r_total = exp_config['params']['r_total']
        for ratio in exp_config['params']['pilot_ratios']:
            r0 = int(r_total * ratio)
            r1 = r_total - r0
            for s_name in exp_config['scenarios']:
                for sim_id in range(1, config.N_SIM + 1):
                    specific_params = {'r': {'r0': r0, 'r1': r1}, 'k_folds': 2, 'pilot_ratio': round(ratio, 2)}
                    tasks.append((exp_name, s_name, 'OS', sim_id, specific_params))

    elif exp_name == "experiment_2_main_comparison":
        r_params = {'r0': exp_config['params']['r0'], 'r1': exp_config['params']['r1']}
        for s_name in exp_config['scenarios']:
            for m_name in exp_config['methods']:
                for sim_id in range(1, config.N_SIM + 1):
                    # FULL method doesn't need 'r' parameter since it uses all data
                    specific_params = {'k_folds': exp_config['params']['k_folds']} if m_name == 'FULL' else {'r': r_params, 'k_folds': exp_config['params']['k_folds']}
                    tasks.append((exp_name, s_name, m_name, sim_id, specific_params))

    elif exp_name == "experiment_3_robustness_check":
        r_params = {'r0': exp_config['params']['r0'], 'r1': exp_config['params']['r1']}
        for misspec in exp_config['params']['misspecification_scenarios']:
             for sim_id in range(1, config.N_SIM + 1):
                specific_params = {'r': r_params, 'k_folds': exp_config['params']['k_folds'], 'misspecification': misspec}
                tasks.append((exp_name, exp_config['scenarios'][0], 'OS', sim_id, specific_params))
    
    print(f"Total simulation tasks: {len(tasks)}")
    print(f"Results will be saved in: '{exp_config['base_dir']}'")
    
    # Run tasks in parallel
    if tasks:
        Parallel(n_jobs=-1)(tqdm(map(delayed(run_single_replication), tasks), total=len(tasks), desc=f"Progress - {exp_name}"))
    
    print(f"\n✅ {exp_name} complete.")

print("\n\nAll simulation experiments have finished. Running final evaluation...")
evaluation.main()


🚀 === Running: experiment_1_pilot_allocation ===
   Investigates the effect of pilot sample allocation on OS-DML performance.
Total simulation tasks: 900
Results will be saved in: './simulation_results/exp1_pilot_allocation'


Progress - experiment_1_pilot_allocation:   0%|          | 0/900 [00:00<?, ?it/s]


✅ experiment_1_pilot_allocation complete.

🚀 === Running: experiment_2_main_comparison ===
   Comprehensive performance comparison of all methods across all DGPs.
Total simulation tasks: 800
Results will be saved in: './simulation_results/exp2_main_comparison'


Progress - experiment_2_main_comparison:   0%|          | 0/800 [00:00<?, ?it/s]


✅ experiment_2_main_comparison complete.

🚀 === Running: experiment_3_robustness_check ===
   Verifies bias reduction and double robustness of OS-DML under the OBS-C DGP.
Total simulation tasks: 200
Results will be saved in: './simulation_results/exp3_robustness_check'


Progress - experiment_3_robustness_check:   0%|          | 0/200 [00:00<?, ?it/s]


✅ experiment_3_robustness_check complete.


All simulation experiments have finished. Running final evaluation...

--- Evaluating: experiment_1_pilot_allocation ---
Generating summary table...

Summary table saved to ./analysis_results/experiment_1_pilot_allocation\summary_table.csv

--- experiment_1_pilot_allocation Summary Table ---
 pilot_ratio method scenario  Coverage  CI_Width    Bias   RMSE  Runtime
      0.1000     OS    OBS-S    0.7600    0.1073 -0.0387 0.0457  26.6950
      0.1000     OS    RCT-S    0.9000    0.0678 -0.0032 0.0216  12.9623
      0.2000     OS    OBS-S    0.8000    0.1014 -0.0266 0.0365  21.1485
      0.2000     OS    RCT-S    0.9400    0.0640  0.0031 0.0199  14.8048
      0.3000     OS    OBS-S    0.8200    0.0958 -0.0214 0.0354  22.4717
      0.3000     OS    RCT-S    0.9200    0.0621  0.0017 0.0164  10.2844
      0.4000     OS    OBS-S    0.8800    0.0980 -0.0166 0.0311  26.3398
      0.4000     OS    RCT-S    0.9200    0.0638 -0.0012 0.0208  11.2117
     